## Tello 드론 Mission Pad — go / curve / jump 파이썬 코드 (고급)

---

## 📋 사전 준비
```bash
프로젝트 환경 만들기
  mkdir missionpad         : 프로젝트폴더생성
  cd missionpad            : 프로젝트폴더로 이동
  py --list                : 파이썬 버전 목록 확인
  py -3.14 -m venv venv    : 파이썬 가상환경 만들기
  venv\Scripts\activate    : 가상환경 시작( 종료는 : deactivate )
  pip install jupyterlab   : 주피터랩 패키지 설치
  pip install djitellopy   : DJI드론 API 설치
  jupyter lab --port=10888 : 주피터랩 시작
```

```
Mission Pad ID 규칙:
  mid 1~8 = m1~m8 (Tello SDK 기준)
  go/curve : 현재 위치 → 미션패드 기준 좌표로 이동
  jump     : 미션패드1 → 미션패드2 간 이동
```

---  


## 📊 명령어 파라미터 요약
<span style="position:left;display:inline-block;">
<table style="font-size:17px;">
    <tr align=center>
        <td>명령</td>
        <td>파라미터</td>
        <td>제한값</td>
    </tr>
    <tr>
        <td align=center>go</td>
        <td>x y z speed mid<br>미션패드 기준 좌표로 직선 이동</td>
        <td>xyz: -500~500<br>speed: 10~100</td>
    </tr>
    <tr>
        <td align=center>curve</td>
        <td>x1 y1 z1  x2 y2 z2  speed mid<br>경유점(x1y1z1) → 도착점(x2y2z2) 곡선 비행</td>
        <td>xyz: -500~500<br>speed: 10~60</td>
    </tr>
    <tr>
        <td align=center>jump</td>
        <td>x y z speed yaw mid1 mid2<br>미션패드1 → 미션패드2 간 이동</td>
        <td>xyz: -500~500<br>yaw: 0~360</td>
    </tr>
    <tr colspan=3><td>○ mid (Mission Pad ID) : 1~8 (m1~m8)<br>
○ 단위 : 좌표(cm), 속도(cm/s), 각도(도)</td></tr>
</table>
</span>

---


## 🟢 고급 (Advanced) — 클래스 기반 + 상태 모니터링 + 자동화


In [ ]:
from djitellopy import Tello
import time
import logging
import threading
from dataclasses import dataclass
from typing import Optional

# ── 로깅 설정 ───────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s'
)
log = logging.getLogger("TelloMission")


# ── 미션 명령 데이터 클래스 ─────────────────────────
@dataclass
class GoCommand:
    x: int; y: int; z: int
    speed: int = 30
    mid: int = 1

@dataclass
class CurveCommand:
    x1: int; y1: int; z1: int
    x2: int; y2: int; z2: int
    speed: int = 30
    mid: int = 1

@dataclass
class JumpCommand:
    x: int; y: int; z: int
    speed: int = 30
    yaw: int = 0
    mid1: int = 1
    mid2: int = 2


# ── 메인 미션 클래스 ────────────────────────────────
class TelloMissionRunner:

    MIN_BATTERY = 20          # 최소 배터리 %
    DETECT_TIMEOUT = 5        # 미션패드 감지 대기 (초)

    def __init__(self):
        self.drone = Tello()
        self._monitoring = False
        self._monitor_thread: Optional[threading.Thread] = None

    # ── 연결 및 초기화 ────────────────────────────
    def connect(self):
        self.drone.connect()
        battery = self.drone.get_battery()
        log.info(f"연결 완료 | 배터리: {battery}%")

        if battery < self.MIN_BATTERY:
            raise RuntimeError(f"배터리 부족: {battery}% (최소 {self.MIN_BATTERY}% 필요)")

        self.drone.enable_mission_pads()
        self.drone.set_mission_pad_detection_direction(2)  # 아래+앞 동시 감지
        log.info("미션패드 감지 활성화 완료")

    # ── 실시간 텔레메트리 모니터링 (별도 스레드) ──
    def start_monitoring(self):
        self._monitoring = True
        self._monitor_thread = threading.Thread(
            target=self._monitor_loop, daemon=True
        )
        self._monitor_thread.start()
        log.info("텔레메트리 모니터링 시작")

    def stop_monitoring(self):
        self._monitoring = False
        if self._monitor_thread:
            self._monitor_thread.join(timeout=2)

    def _monitor_loop(self):
        while self._monitoring:
            try:
                log.info(
                    f"[텔레메트리] "
                    f"배터리:{self.drone.get_battery()}% | "
                    f"높이:{self.drone.get_height()}cm | "
                    f"온도:{self.drone.get_temperature()}°C | "
                    f"미션패드:{self.drone.get_mission_pad_id()}"
                )
            except Exception as e:
                log.warning(f"텔레메트리 오류: {e}")
            time.sleep(2)

    # ── 미션패드 감지 대기 ────────────────────────
    def wait_for_pad(self, mid: int) -> bool:
        log.info(f"미션패드 {mid} 감지 대기 중...")
        start = time.time()
        while time.time() - start < self.DETECT_TIMEOUT:
            detected = self.drone.get_mission_pad_id()
            if detected == mid:
                log.info(f"✅ 미션패드 {mid} 감지 성공")
                return True
            time.sleep(0.3)
        log.warning(f"⚠️ 미션패드 {mid} 감지 실패 (타임아웃)")
        return False

    # ── go 명령 ───────────────────────────────────
    def execute_go(self, cmd: GoCommand):
        log.info(f"[GO] x={cmd.x} y={cmd.y} z={cmd.z} "
                 f"speed={cmd.speed} mid={cmd.mid}")
        self._validate_go(cmd)

        if not self.wait_for_pad(cmd.mid):
            raise RuntimeError(f"미션패드 {cmd.mid} 미감지 — go 취소")

        self.drone.go_xyz_speed_mid(
            cmd.x, cmd.y, cmd.z,
            cmd.speed, cmd.mid
        )
        log.info("[GO] 완료")
        time.sleep(2)

    # ── curve 명령 ────────────────────────────────
    def execute_curve(self, cmd: CurveCommand):
        log.info(f"[CURVE] 경유({cmd.x1},{cmd.y1},{cmd.z1}) "
                 f"→ 도착({cmd.x2},{cmd.y2},{cmd.z2}) "
                 f"speed={cmd.speed} mid={cmd.mid}")
        self._validate_curve(cmd)

        if not self.wait_for_pad(cmd.mid):
            raise RuntimeError(f"미션패드 {cmd.mid} 미감지 — curve 취소")

        self.drone.curve_xyz_speed_mid(
            cmd.x1, cmd.y1, cmd.z1,
            cmd.x2, cmd.y2, cmd.z2,
            cmd.speed, cmd.mid
        )
        log.info("[CURVE] 완료")
        time.sleep(2)

    # ── jump 명령 ─────────────────────────────────
    def execute_jump(self, cmd: JumpCommand):
        log.info(f"[JUMP] mid{cmd.mid1} → mid{cmd.mid2} | "
                 f"x={cmd.x} y={cmd.y} z={cmd.z} "
                 f"yaw={cmd.yaw}")

        if not self.wait_for_pad(cmd.mid1):
            raise RuntimeError(f"출발 미션패드 {cmd.mid1} 미감지 — jump 취소")

        self.drone.go_xyz_speed_yaw_mid(
            cmd.x, cmd.y, cmd.z,
            cmd.speed, cmd.yaw,
            cmd.mid1, cmd.mid2
        )
        log.info("[JUMP] 완료")
        time.sleep(2)

    # ── 파라미터 유효성 검사 ──────────────────────
    def _validate_go(self, cmd: GoCommand):
        for val, name in [(cmd.x,'x'),(cmd.y,'y'),(cmd.z,'z')]:
            if not (-500 <= val <= 500):
                raise ValueError(f"go {name} 범위 초과: {val} (-500~500)")
        if not (10 <= cmd.speed <= 100):
            raise ValueError(f"go speed 범위 초과: {cmd.speed} (10~100)")
        if not (1 <= cmd.mid <= 8):
            raise ValueError(f"mid 범위 초과: {cmd.mid} (1~8)")

    def _validate_curve(self, cmd: CurveCommand):
        coords = [cmd.x1,cmd.y1,cmd.z1,cmd.x2,cmd.y2,cmd.z2]
        for val in coords:
            if not (-500 <= val <= 500):
                raise ValueError(f"curve 좌표 범위 초과: {val}")
        if not (10 <= cmd.speed <= 60):
            raise ValueError(f"curve speed 범위 초과: {cmd.speed} (10~60)")

    # ── 미션 실행 ─────────────────────────────────
    def run(self):
        try:
            self.connect()
            self.drone.takeoff()
            log.info("이륙 완료")
            time.sleep(3)

            self.start_monitoring()

            # ① go 명령
            self.execute_go(GoCommand(
                x=0, y=0, z=80,
                speed=30, mid=1
            ))

            # ② curve 명령
            self.execute_curve(CurveCommand(
                x1=60,  y1=0,  z1=80,
                x2=60, y2=60, z2=80,
                speed=30, mid=1
            ))

            # ③ jump 명령 (미션패드 1 → 2)
            self.execute_jump(JumpCommand(
                x=0, y=0, z=80,
                speed=30, yaw=0,
                mid1=1, mid2=2
            ))

            log.info("✅ 전체 미션 완료")

        except Exception as e:
            log.error(f"❌ 미션 오류: {e}")
        finally:
            self.stop_monitoring()
            log.info("착륙 시도 중...")
            self.drone.land()
            self.drone.disable_mission_pads()
            self.drone.end()
            log.info("드론 종료 완료")


# ── 실행 ────────────────────────────────────────────
if __name__ == '__main__':
    mission = TelloMissionRunner()
    mission.run()